# Homework 2. Programming in Haskell
i.dzhabbarov@innopolis.university


[@iliyasone](https://t.me/iliyasone)

### 2.1 Forward Mode Automatic Differentiation

In [1]:
data Forward a = Forward a a
    deriving (Show)
    
lift :: Num a => a -> Forward a
lift x = Forward x 1

putStr "OK"

OK

In [2]:
example_f_dy :: Floating a => a -> Forward a
example_f_dy y = Forward (y * sin y) (y * cos y + sin y)
example_f :: Floating a => Forward a -> Forward a
example_f (Forward y y') = Forward (y * sin y) ((y * cos y + sin y) * y')


example_f_dy (pi/3)
example_f (lift (pi/3))


Line 1: Use camelCase
Found:
example_f_dy :: Floating a => a -> Forward a
Why not:
exampleFDy :: Floating a => a -> Forward aLine 2: Use camelCase
Found:
example_f_dy y = ...
Why not:
exampleFDy y = ...Line 3: Use camelCase
Found:
example_f :: Floating a => Forward a -> Forward a
Why not:
exampleF :: Floating a => Forward a -> Forward aLine 4: Use camelCase
Found:
example_f (Forward y y') = ...
Why not:
exampleF (Forward y y') = ...

Forward 0.9068996821171088 1.3896241793827375

Forward 0.9068996821171088 1.3896241793827375

#### Exercise 2.1.1 

In [3]:
example_g :: Num a => Forward a -> Forward a
example_g (Forward y y') = Forward ((y - 1) * (y - 1) + 1) ((2 * y - 2) * y')

Line 1: Use camelCase
Found:
example_g :: Num a => Forward a -> Forward a
Why not:
exampleG :: Num a => Forward a -> Forward aLine 2: Use camelCase
Found:
example_g (Forward y y') = ...
Why not:
exampleG (Forward y y') = ...

In [4]:
example_g (lift 3)
example_g (example_g (lift 3))
example_f (example_g (lift 3))

Forward 5 4

Forward 17 32

Forward (-4.794621373315692) 1.8375466106119709

#### Exercise 2.1.2

In [5]:
constant :: Num a => a -> Forward a
constant a = Forward a 0


constantF :: Num a => a -> Forward a -> Forward a
constantF a _ = Forward a 0

linear :: Num a => a -> a -> Forward a -> Forward a
linear a b (Forward y y') = Forward (a*y+ b) (a*y')

In [6]:
constant 4
constantF 4 (lift 123)
linear 2 (-3) (lift 1)
linear 2 (-3) (lift 6)
linear 2 (-3) (example_f (lift 6))

Forward 4 0

Forward 4 0

Forward (-1) 2

Forward 9 2

Forward (-6.35298597838711) 10.96321244340654

#### Exercise 2.1.3 

In [7]:
diff :: Num a => (Forward a -> Forward a) -> a -> a
diff f a = y'
  where
    Forward _ y' = f (Forward a 1)

In [8]:
diff (linear 2 (-3)) 7.0
diff example_f (pi/3)
diff example_g 4.0

2.0

1.3896241793827375

6.0

#### Exercise 2.1.4

In [9]:
plusF :: Num a => Forward a -> Forward a -> Forward a
plusF (Forward x x') (Forward y y') = Forward (x + y) (x' + y')

timesF :: Num a => Forward a -> Forward a -> Forward a
timesF (Forward x x') (Forward y y') = Forward (x * y) (x * y' + y * x')

negateF :: Num a => Forward a -> Forward a
negateF (Forward x x') = Forward (-x) (-x')

absoluteF :: Num a => Forward a -> Forward a
absoluteF (Forward x x') = Forward (abs x) (signum x * x')


signumF :: Num a => Forward a -> Forward a
signumF (Forward x x') = Forward (signum x) 0

recipF :: Fractional a => Forward a -> Forward a
recipF (Forward x x') = Forward (recip x) (- recip (x * x) * x')

In [10]:
instance Num a => Num (Forward a) where
    fromInteger n = constant (fromInteger n)
    (+) = plusF
    (*) = timesF
    negate = negateF
    abs = absoluteF
    signum = signumF
instance Fractional a => Fractional (Forward a) where
    fromRational x = constant (fromRational x)
    recip = recipF

In [11]:
diff (\x -> x^3 + 2 * x + 1) 1
diff (\x -> (x `timesF` x `timesF` x) `plusF` (constant 2 `timesF` x) `plusF` 1) 1

5

5

In [12]:
diff (\x -> signum x * (x - 2) * abs x) 4
diff (\x -> signum x * (x - 2) * abs x) (-2)
diff (\x -> x ^ 10) 2
diff (\x -> 1 / x) 2
diff (\x -> x^2 / x) 2
diff (\x -> (abs x - 1)^2 / (x + 8)) (-3)

Line 3: Avoid lambda using `infix`
Found:
(\ x -> x ^ 10)
Why not:
(^ 10)Line 4: Avoid lambda
Found:
(\ x -> 1 / x)
Why not:
(1 /)

6

-6

5120

-0.25

1.0

-0.9600000000000001

#### Exercise 2.1.5

In [13]:
expF :: Floating a => Forward a -> Forward a
expF (Forward x x') = Forward (exp x) (exp x * x') 

logF :: Floating a => Forward a -> Forward a
logF (Forward x x') = Forward (log x) (x' / x)

sinF :: Floating a => Forward a -> Forward a
sinF (Forward x x') = Forward (sin x) (cos x * x')

cosF :: Floating a => Forward a -> Forward a
cosF (Forward x x') = Forward (cos x) (-sin x * x')

asinF :: Floating a => Forward a -> Forward a
asinF (Forward x x') = Forward (asin x) (x' / sqrt(1 - x^2))

acosF :: Floating a => Forward a -> Forward a
acosF (Forward x x') = Forward (acos x) (-x' / sqrt(1 - x^2))

atanF :: Floating a => Forward a -> Forward a
atanF (Forward x x') = Forward (atan x) (x' / (1 + x^2))

In [14]:
instance Floating a => Floating (Forward a) where
    pi = constant pi
    exp = expF
    log = logF
    sin = sinF
    cos = cosF
    asin = asinF
    acos = acosF
    atan = atanF

In [15]:
diff (\x -> sin (asin x)) 0.1
diff (\x -> acos (cos x)) 0.1
diff (\x -> atan (tan x)) 0.1

Line 1: Avoid lambda
Found:
\ x -> sin (asin x)
Why not:
sin . asinLine 2: Avoid lambda
Found:
\ x -> acos (cos x)
Why not:
acos . cosLine 3: Avoid lambda
Found:
\ x -> atan (tan x)
Why not:
atan . tan

1.0

1.0000000000000053

1.0

In [16]:
f x = x * sin x
f (pi/2)
diff f (pi/2)
diff (diff f) (pi/2)
diff (diff (diff f)) (pi/2)

1.5707963267948966

1.0

-1.5707963267948966

-3.0

#### Exercise 2.1.6 

**1.**

Let's remind definitions:

```haskell
data Forward a = Forward a a deriving (Show)

diff :: Num a => (Forward a -> Forward a) -> a -> a
diff f a = y'
  where
    Forward _ y' = f (Forward a 1)
```

Also, our function `f` is defined as:

```haskell
f :: Floating a => a -> a
f x = x * sin x
```

**(a) `diff f (pi/2) :: Double`**

We are computing the first derivative of `f` at `pi/2`, here's how the types are instantiated:

- **In `diff`:**
  - Type variable `a` is instantiated to `Double`.
  - So, `diff` has type `(Forward Double -> Forward Double) -> Double -> Double`

- **Function `f`:**
  - Originally of type `Floating a => a -> a`.
  - Since `Forward Double` is an instance of `Floating`, `f` can be used as `Forward Double -> Forward Double`

- **Conclusion:**
  - `diff f (pi/2)` computes the derivative of `f` at `pi/2`, resulting in a `Double`

**(b) `diff (diff f) (pi/2) :: Double`**

Here, we compute the second derivative:

- **First `diff f`:**
  - `diff f` has type `Double -> Double` from part (a)

- **In the outer `diff`:**
  - Type variable `a` is instantiated to `Double`
  - However, `diff` expects a function of type `Forward a -> Forward a`
  - We need `diff f` to be of type `Forward Double -> Forward Double`

- **But `diff f` is `Double -> Double`.**

To resolve this, Haskell leverages its polymorphic type system and the nested `Forward` types:

- **Instantiate `a` in the outer `diff` to `Forward Double`:**
  - Now, `diff` has type `(Forward (Forward Double) -> Forward (Forward Double)) -> Forward Double -> Forward Double`.
  - `diff f` now operates over `Forward Double`, making its type `Forward Double -> Forward Double`.

- **Conclusion:**
  - `diff (diff f)` computes the second derivative, and the types align correctly due to nesting of `Forward` types.

**(c) `diff (diff (diff f)) (pi/2) :: Double`**

For the third derivative:

- **Instantiate `a` in the outermost `diff` to `Forward (Forward Double)`:**
  - Outer `diff` has type `(Forward (Forward (Forward Double)) -> Forward (Forward (Forward Double))) -> Forward (Forward Double) -> Forward (Forward Double)`.
  - `diff (diff f)` now has type `Forward (Forward Double) -> Forward (Forward Double)`.

- **Conclusion:**
  - By nesting `Forward` types, we compute higher derivatives.


**2.0**

Reminder:

```haskell
plusF (Forward x x') (Forward y y') = Forward (x + y) (x' + y')

timesF (Forward x x') (Forward y y') = Forward (x * y) (x * y' + y * x')

sinF (Forward x x') = Forward (sin x) (cos x * x')

cosF (Forward x x') = Forward (cos x) (- sin x * x')
```

Our goal is to prove:

**(a) `diff f x = x * cos x + sin x`**

- **Compute `f (Forward x 1)`:**
  - `f (Forward x 1) = (Forward x 1) * sin (Forward x 1)`

- **Compute `sinF (Forward x 1)`:**
  - `sinF (Forward x 1) = Forward (sin x) (cos x * 1) = Forward (sin x) (cos x)`

- **Compute `timesF (Forward x 1) (Forward (sin x) (cos x))`:**
  - Value: `x * sin x`
  - Derivative: `x * cos x + sin x * 1 = x * cos x + sin x`

- **Therefore, `f (Forward x 1) = Forward (x * sin x) (x * cos x + sin x)`.**

- **Conclusion:**
  - `diff f x = x * cos x + sin x`

**(b) `diff (diff f) x = x * (- sin x) + 2 * cos x`**

- **From part (a), we have `f' x = x * cos x + sin x`.**

- **Compute `f' (Forward x 1)`:**
  - `f' (Forward x 1) = timesF (Forward x 1) (cosF (Forward x 1)) + sinF (Forward x 1)`

- **Compute `cosF (Forward x 1)`:**
  - `cosF (Forward x 1) = Forward (cos x) (- sin x * 1) = Forward (cos x) (- sin x)`

- **Compute `timesF (Forward x 1) (Forward (cos x) (- sin x))`:**
  - Value: `x * cos x`
  - Derivative: `x * (- sin x) + cos x * 1 = - x * sin x + cos x`

- **Compute `sinF (Forward x 1)`:**
  - `sinF (Forward x 1) = Forward (sin x) (cos x)`

- **Compute `plusF`:**
  - Value: `x * cos x + sin x` (as before)
  - Derivative: `(- x * sin x + cos x) + cos x = - x * sin x + 2 * cos x`

- **Conclusion:**
  - `diff (diff f) x = - x * sin x + 2 * cos x`

#### Exercise 2.1.7

In [65]:
newton :: Floating a => (Forward a -> Forward a) -> a -> [a]

newtonNoFirst f x0 = x1 : newtonNoFirst f x1
    where
        Forward f_x0 f_x0' = f (Forward x0 1)
        x1 = x0 - f_x0 / f_x0'
        
newton f x0 = x0 : newtonNoFirst f x0

In [66]:
mapM_ print $ take 10 $ newton (\x -> (x - 3)^2 - 16) 4

4.0
11.5
8.191176470588236
7.136664722546242
7.002257524798522
7.000000636692939
7.000000000000051
7.0
7.0
7.0

In [19]:
data Equation a = a :=: a
infix 1 :=:

#### Exercise 2.1.8

In [69]:
solve :: (Ord a, Floating a) => (Forward a -> Equation (Forward a)) -> Maybe a

solve f = helper 0 0.2024
    where 
        helper iter x
            | iter >= 1000 = Nothing
            | abs(x - x_next) < 1e-3 = Just x_next
            | otherwise = helper (iter + 1) x_next
                where
                    (lhs :=: rhs) = f (Forward x 1)
                    x_next = head $ tail $ newton (\y -> lhs - rhs) x
                    

In [70]:
solve (\x -> x * sin x :=: 1)
map (\n -> solve (\x -> x^n :=: 1024)) [2,5,10,100]

Just 2.772604795692667

[Just 32.00000000035865,Just 4.000000000806468,Just 2.0000003470909324,Nothing]

### 2.2 Vectors, Gradients, and Gradient Descent

In [22]:
data V2 a = V2 a a
    deriving (Show)

#### Exercise 2.2.1 

In [23]:
scaleV2 :: Num a => a -> V2 a -> V2 a
scaleV2 a (V2 x y) = V2 (a * x) (a * y)


addV2 :: Num a => V2 a -> V2 a -> V2 a
addV2 (V2 x1 y1) (V2 x2 y2) = V2 (x1 + x2) (y1 + y2)

negateV2 :: Num a => V2 a -> V2 a
negateV2 (V2 x y) = V2 (-x) (-y)

lengthV2 :: Floating a => V2 a -> a
lengthV2 (V2 x y) = sqrt (x*x + y*y)

In [24]:
scaleV2 0.5 (V2 3 4)
addV2 (V2 1 2) (V2 3 4)
negateV2 (V2 3 4)
lengthV2 (V2 3 4)

V2 1.5 2.0

V2 4 6

V2 (-3) (-4)

5.0

In [26]:
example_h :: Num a => V2 a -> a
example_h (V2 x y) = x^2 + x * y^2

dx :: Num a => V2 a -> V2 (Forward a)
dx (V2 x y) = V2 (lift x) (constant y)

dy :: Num a => V2 a -> V2 (Forward a)
dy (V2 x y) = V2 (constant x) (lift y)

Line 1: Use camelCase
Found:
example_h :: Num a => V2 a -> a
Why not:
exampleH :: Num a => V2 a -> aLine 2: Use camelCase
Found:
example_h (V2 x y) = ...
Why not:
exampleH (V2 x y) = ...

#### Exercise 2.2.2 

In [43]:
pdiff :: Num a
    => (V2 a -> V2 (Forward a)) -- Direction of differentiation (dx or dy).
    -> (V2 (Forward a) -> Forward a) -- Vector-to-scalar function.
    -> V2 a -- The vector at which to compute the partial derivative.
    -> a

pdiff direction vector_to_scalar v0 = r
    where
        Forward _ r = vector_to_scalar (direction v0)

In [44]:
example_h (V2 2 3)
pdiff dx example_h (V2 2 3)
pdiff dy example_h (V2 2 3)

22

13

12

#### Exercise 2.2.3

In [48]:
grad :: Num a => (V2 (Forward a) -> Forward a) -> V2 a -> V2 a
grad f v0 =  V2 (pdiff dx f v0 ) (pdiff dy f v0 )

In [49]:
grad (\(V2 x y) -> (x - 1)^2 * (y - 2)^3) (V2 2 5)
grad (\(V2 x y) -> sin x * cos y) (V2 (pi/3) (pi/3))

V2 54 27

V2 0.2500000000000001 (-0.7499999999999999)

#### Exercise 2.2.4

In [63]:
naiveGradientDescent :: Floating a 
    => a 
    -> (V2 (Forward a) -> Forward a) 
    -> V2 a 
    -> [V2 a]
    
    
naiveGradientDescentNoFirst gamma vector_to_scalar v0 = v1 : naiveGradientDescentNoFirst gamma vector_to_scalar v1
    where
        v1 = addV2 v0 $ scaleV2 gamma $ negateV2 $ grad vector_to_scalar v0
        
naiveGradientDescent gamma vector_to_scalar v0 = v0 : naiveGradientDescentNoFirst gamma vector_to_scalar v0

In [64]:
mapM_ print $ take 10 $ naiveGradientDescent 0.1 (\(V2 x y) -> x^2 + y^2) (V2 1 2)

V2 1.0 2.0
V2 0.8 1.6
V2 0.64 1.28
V2 0.512 1.024
V2 0.4096 0.8192
V2 0.32768 0.65536
V2 0.26214400000000004 0.5242880000000001
V2 0.20971520000000005 0.4194304000000001
V2 0.16777216000000003 0.33554432000000006
V2 0.13421772800000004 0.26843545600000007

####  Exercise 2.2.5

In [76]:
instance Num a => Num (V2 a) where
    (+) = addV2
    negate = negateV2

In [86]:
liftV2 :: Num a => V2 a -> V2 (Forward a)
liftV2 (V2 x y) = V2 (constant x) (constant y)

value :: Forward a -> a
value (Forward val _) = val


backtrackingLineSearch
    :: (Floating a, Ord a)
    => a                             -- The control parameter c.
    -> (V2 (Forward a) -> Forward a) -- The function f.
    -> V2 a                          -- Current vector v.
    -> a                             -- Initial approximation of γ.
    -> a
    
backtrackingLineSearch c f v0 gamma
    | f_v_new - f_v0 <= -c * gamma * lengthV2 d = gamma
    | otherwise = backtrackingLineSearch c f v0 (gamma / 2)
    where
        d = negateV2 (grad f v0)         
        v_new = v0 `addV2` scaleV2 gamma d 
        f_v_new = value (f (liftV2 v_new)) 
        f_v0 = value (f (liftV2 v0))       

In [88]:
backtrackingLineSearch 0.5 (\(V2 x y) -> x^2 + y^2) (V2 1 3) 1
backtrackingLineSearch 0.5 (\(V2 x y) -> 3 * x * exp (- x^2 - y^2)) (V2 0 0) 0.9

0.5

0.225

#### Exercise 2.2.6

In [90]:
gradientDescent :: (Floating a, Ord a) => a -> (V2 (Forward a) -> Forward a) -> V2 a -> [V2 a]
gradientDescent gamma f v0 = v0 : gradientDescent' gamma v0
  where
    gradientDescent' gamma_n v_n = v_n_plus1 : gradientDescent' gamma_n_plus1 v_n_plus1
      where
        grad_f_vn = grad f v_n                 
        d = negateV2 grad_f_vn                 
        gamma_n' = backtrackingLineSearch 0.5 f v_n gamma_n  
        v_n_plus1 = v_n `addV2` scaleV2 gamma_n' d           
        gamma_n_plus1 = 2 * gamma_n'                        


In [91]:
mapM_ print $ take 10 $ gradientDescent 0.01 (\(V2 x y) -> x^2 + y^2) (V2 9 3)

V2 9.0 3.0
V2 8.82 2.94
V2 8.4672 2.8224
V2 7.789824 2.596608
V2 6.54345216 2.1811507199999998
V2 4.4495474688000005 1.4831824895999999
V2 1.6018370887680002 0.533945696256
V2 (-0.44851438485504014) (-0.14950479495168)
V2 (-0.16146517854781445) (-5.382172618260481e-2)
V2 (-0.16146517854781445) (-5.382172618260481e-2)